# Comparaison EWC vs HDC vs Fine-tuning — Dataset 2

**Sprint 2 — S2-04** | CL-Embedded | ISAE-SUPAERO × ENAC × Edge Spectrum  
**Dataset** : Industrial Equipment Monitoring — 3 domaines (Pump → Turbine → Compressor)  
**Expériences** : `exp_001_ewc_dataset2` (S1-09) et `exp_002_hdc_dataset2` (S2-03)

Ce notebook est exécutable de bout en bout sans intervention manuelle :
```
jupyter nbconvert --to notebook --execute notebooks/02_baseline_comparison.ipynb
```

In [1]:
# Cell 0 : Setup
from pathlib import Path
import json
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")  # backend non-interactif pour nbconvert
import matplotlib.pyplot as plt
import pandas as pd

import sys

# Repositionner au root du dépôt quelle que soit la CWD d'exécution
_cwd = Path(".").resolve()
if _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.memory_profiler import compare_models_memory

# Chemins des expériences
EXP_EWC = Path("experiments/exp_001_ewc_dataset2/results")
EXP_HDC = Path("experiments/exp_002_hdc_dataset2/results")

# Répertoires de sortie figures organisés par type
FIGURES_CL_EVAL = Path("notebooks/figures/cl_evaluation")
FIGURES_MODEL_VIZ = Path("notebooks/figures/model_viz")
FIGURES_CL_EVAL.mkdir(parents=True, exist_ok=True)
FIGURES_MODEL_VIZ.mkdir(parents=True, exist_ok=True)

DOMAINS = ["Pump", "Turbine", "Compressor"]

# Vérification que les fichiers requis existent
required = [
    EXP_EWC / "metrics.json",
    EXP_EWC / "acc_matrix_ewc.npy",
    EXP_EWC / "memory_report.json",
    EXP_HDC / "metrics.json",
    EXP_HDC / "acc_matrix_hdc.npy",
    EXP_HDC / "memory_report.json",
]
for p in required:
    assert p.exists(), f"Fichier manquant : {p}. Lancer S1-09 et S2-03 d'abord."

print("Tous les fichiers requis sont présents.")

Tous les fichiers requis sont présents.


In [2]:
# Cell 1 : Chargement des résultats

# --- EWC (exp_001) ---
with open(EXP_EWC / "metrics.json") as f:
    data_ewc = json.load(f)

# Structure : data_ewc["cl_metrics"]["ewc" | "naive" | "joint" | "memory"]
m_ewc   = data_ewc["cl_metrics"]["ewc"]    # métriques CL EWC Online
m_naive = data_ewc["cl_metrics"]["naive"]  # métriques fine-tuning naïf
m_joint = data_ewc["cl_metrics"]["joint"]  # métriques joint training
mem_ewc_inline = data_ewc["cl_metrics"]["memory"]  # profil mémoire inline

# Enrichir m_ewc avec les données mémoire (pour le tableau comparatif)
m_ewc["ram_peak_bytes"]       = mem_ewc_inline["forward"]["ram_peak_bytes"]
m_ewc["inference_latency_ms"] = mem_ewc_inline["forward"]["inference_latency_ms"]
m_ewc["n_params"]             = mem_ewc_inline["forward"]["n_params"]

# Matrices d'accuracy
acc_ewc   = np.load(EXP_EWC / "acc_matrix_ewc.npy",   allow_pickle=True)
acc_naive = np.load(EXP_EWC / "acc_matrix_naive.npy", allow_pickle=True)
acc_joint = np.load(EXP_EWC / "acc_matrix_joint.npy", allow_pickle=True)

# --- HDC (exp_002) ---
with open(EXP_HDC / "metrics.json") as f:
    data_hdc = json.load(f)

# Structure : data_hdc["cl_metrics"] contient directement aa, af, bwt, ram_peak_bytes, ...
m_hdc = data_hdc["cl_metrics"]

acc_hdc = np.load(EXP_HDC / "acc_matrix_hdc.npy", allow_pickle=True)

# --- Memory reports (pour compare_models_memory) ---
with open(EXP_EWC / "memory_report.json") as f:
    mem_report_ewc = json.load(f)
with open(EXP_HDC / "memory_report.json") as f:
    mem_report_hdc = json.load(f)

print("Métriques EWC  :", {k: round(v, 4) for k, v in m_ewc.items() if isinstance(v, float)})
print("Métriques HDC  :", {k: round(v, 4) for k, v in m_hdc.items() if isinstance(v, float)})
print("Métriques Naïf :", {k: round(v, 4) for k, v in m_naive.items() if isinstance(v, float)})

Métriques EWC  : {'aa': 0.9824, 'af': 0.001, 'bwt': 0.0, 'fwt': 0.0, 'inference_latency_ms': 0.0358}
Métriques HDC  : {'aa': 0.8698, 'af': 0.0, 'bwt': 0.0019, 'fwt': 0.0, 'inference_latency_ms': 0.0476}
Métriques Naïf : {'aa': 0.9811, 'af': 0.0, 'bwt': 0.001, 'fwt': 0.0}


In [3]:
# Cell 2 : Tableau comparatif AA / AF / BWT / RAM / Latence

def _fmt(val, fmt):
    """Formate val avec fmt, ou 'N/A' si None/NaN."""
    if val is None:
        return "N/A"
    try:
        if np.isnan(val):
            return "N/A"
    except TypeError:
        pass
    return format(val, fmt)

rows = [
    {
        "Modèle":        "EWC Online (M2)",
        "AA ↑":          _fmt(m_ewc.get("aa"),                    ".4f"),
        "AF ↓":          _fmt(m_ewc.get("af"),                    ".4f"),
        "BWT ↑":         _fmt(m_ewc.get("bwt"),                   ".6f"),
        "RAM peak (B)":  f"{m_ewc.get('ram_peak_bytes', -1):,}",
        "Latence (ms)":  _fmt(m_ewc.get("inference_latency_ms"),  ".4f"),
        "Params":        f"{m_ewc.get('n_params', -1):,}",
    },
    {
        "Modèle":        "HDC (M3)",
        "AA ↑":          _fmt(m_hdc.get("aa"),                    ".4f"),
        "AF ↓":          _fmt(m_hdc.get("af"),                    ".4f"),
        "BWT ↑":         _fmt(m_hdc.get("bwt"),                   ".6f"),
        "RAM peak (B)":  f"{m_hdc.get('ram_peak_bytes', -1):,}",
        "Latence (ms)":  _fmt(m_hdc.get("inference_latency_ms"),  ".4f"),
        "Params":        f"{m_hdc.get('n_params', -1):,}",
    },
    {
        "Modèle":        "Fine-tuning naïf",
        "AA ↑":          _fmt(m_naive.get("aa"),  ".4f"),
        "AF ↓":          _fmt(m_naive.get("af"),  ".4f"),
        "BWT ↑":         _fmt(m_naive.get("bwt"), ".6f"),
        "RAM peak (B)":  "—",
        "Latence (ms)":  "—",
        "Params":        "—",
    },
    {
        "Modèle":        "Joint training (upper bound)",
        "AA ↑":          _fmt(m_joint.get("aa"),  ".4f"),
        "AF ↓":          "N/A",
        "BWT ↑":         "N/A",
        "RAM peak (B)":  "—",
        "Latence (ms)":  "—",
        "Params":        "—",
    },
]

df = pd.DataFrame(rows).set_index("Modèle")
print("Comparaison EWC vs HDC — Dataset 2 (3 domaines)")
print("=" * 75)
print(df.to_string())

Comparaison EWC vs HDC — Dataset 2 (3 domaines)
                                AA ↑    AF ↓     BWT ↑ RAM peak (B) Latence (ms) Params
Modèle                                                                                 
EWC Online (M2)               0.9824  0.0010  0.000012        1,171       0.0358    705
HDC (M3)                      0.8698  0.0000  0.001949       14,504       0.0476  2,048
Fine-tuning naïf              0.9811  0.0000  0.000986            —            —      —
Joint training (upper bound)  0.9811     N/A       N/A            —            —      —


In [4]:
# Cell 3 : Matrices d'accuracy — heatmaps [3 × 3]

def plot_acc_matrix(ax, matrix: np.ndarray, title: str, domains: list) -> None:
    """Affiche une matrice d'accuracy annotée."""
    n = len(domains)
    mat_f = np.array([[float(v) if v is not None else float("nan") for v in row] for row in matrix])
    display_mat = np.where(np.isnan(mat_f), -0.05, mat_f)
    im = ax.imshow(display_mat, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels([f"Task {i+1}\n({d})" for i, d in enumerate(domains)], fontsize=9)
    ax.set_yticklabels([f"After Task {i+1}" for i in range(n)], fontsize=9)
    ax.set_xlabel("Tâche évaluée", fontsize=10)
    ax.set_ylabel("Après entraînement sur", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    for i in range(n):
        for j in range(n):
            val = mat_f[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                        color="black", fontsize=9, fontweight="bold")
            else:
                ax.text(j, i, "N/A", ha="center", va="center", color="gray", fontsize=8)
    plt.colorbar(im, ax=ax, label="Accuracy")


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acc_matrix(axes[0], acc_ewc, "EWC Online (M2)", DOMAINS)
plot_acc_matrix(axes[1], acc_hdc, "HDC (M3)",        DOMAINS)
plt.suptitle("Évolution de l'accuracy par tâche — Dataset 2 (3 domaines)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
out = FIGURES_CL_EVAL / "acc_matrix_comparison.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Figure sauvegardée : {out}")
plt.show()

Figure sauvegardée : notebooks/figures/cl_evaluation/acc_matrix_comparison.png


/tmp/ipykernel_43819/3714085114.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Cell 4 : Prototypes HDC — visualisation des accumulateurs

hdc_checkpoint = Path("experiments/exp_002_hdc_dataset2/checkpoints/hdc_task3_final.npz")

if hdc_checkpoint.exists():
    hdc_state = np.load(hdc_checkpoint)
    prototypes_acc = hdc_state["prototypes_acc"]  # [2, D]
    class_counts   = hdc_state["class_counts"]    # [2]
    D = prototypes_acc.shape[1]
    N_SHOW = min(256, D)

    class_names = ["Normal (0)", "Faulty (1)"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    for ax, proto, name, count in zip(axes, prototypes_acc, class_names, class_counts):
        slice_proto = proto[:N_SHOW]
        bar_colors = ["steelblue" if v >= 0 else "tomato" for v in slice_proto]
        ax.bar(range(N_SHOW), slice_proto, color=bar_colors, width=1)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_title(
            f"Prototype {name} — {int(count)} exemples vus\n"
            f"({N_SHOW}/{D} dimensions affichées)",
            fontsize=10
        )
        ax.set_xlabel("Dimension hypervecteur", fontsize=9)
        ax.set_ylabel("Valeur accumulateur", fontsize=9)

    plt.suptitle("Prototypes HDC après 3 tâches — Dataset 2", fontsize=12, fontweight="bold")
    plt.tight_layout()
    out = FIGURES_MODEL_VIZ / "hdc_prototypes.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Figure sauvegardée : {out}")
    print(f"Classe Normal : {int(class_counts[0])} exemples | Classe Faulty : {int(class_counts[1])} exemples")
    plt.show()
else:
    print(f"Checkpoint HDC non trouvé : {hdc_checkpoint}. Lancer S2-03 d'abord.")

Figure sauvegardée : notebooks/figures/model_viz/hdc_prototypes.png
Classe Normal : 5523 exemples | Classe Faulty : 614 exemples


/tmp/ipykernel_43819/3502518403.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Cell 5 : Comparaison mémoire EWC vs HDC

# Tableau textuel via compare_models_memory
print(compare_models_memory([mem_report_ewc, mem_report_hdc]))
print()

# Graphique comparatif
ewc_fwd = mem_report_ewc["forward"]
hdc_fwd = mem_report_hdc["forward"]

models   = ["EWC Online (M2)", "HDC (M3)"]
ram_vals = [ewc_fwd["ram_peak_bytes"] / 1024, hdc_fwd["ram_peak_bytes"] / 1024]
ewc_upd_ram = mem_report_ewc.get("update", {}).get("ram_peak_bytes_update", 0) / 1024

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#4C72B0", "#DD8452"]
bars = ax.bar(models, ram_vals, color=colors, width=0.4, label="RAM inférence (peak)")

# Tranche supplémentaire EWC update (Fisher)
if ewc_upd_ram > ram_vals[0]:
    ax.bar(["EWC Online (M2)"], [ewc_upd_ram - ram_vals[0]],
           bottom=[ram_vals[0]], color="#4C72B0", alpha=0.4,
           width=0.4, label=f"RAM mise à jour EWC (+{ewc_upd_ram - ram_vals[0]:.1f} Ko)")
    ax.text(0, ewc_upd_ram + 0.5, f"{ewc_upd_ram:.1f} Ko (update)",
            ha="center", fontsize=9, color="#2c5085")

ax.axhline(64, color="red", linestyle="--", linewidth=1.5, label="Budget 64 Ko (STM32N6)")
ax.set_ylabel("RAM (Ko)", fontsize=11)
ax.set_title("Empreinte mémoire — EWC vs HDC (Dataset 2)", fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 70)

for bar, val in zip(bars, ram_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f} Ko", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
out = FIGURES_CL_EVAL / "memory_comparison.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Figure sauvegardée : {out}")
plt.show()

Modèle                   Params   RAM FP32   RAM peak      Latence  Budget OK
-----------------------------------------------------------------------------
EWCMlpClassifier            705       2.8Ko       1.1Ko      0.036ms          ✅
HDCClassifier             2,048      14.0Ko      14.2Ko      0.048ms          ✅

Figure sauvegardée : notebooks/figures/cl_evaluation/memory_comparison.png


/tmp/ipykernel_43819/3797950091.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Cell 6 : Conclusion — Triple gap

from IPython.display import Markdown, display

aa_ewc  = m_ewc.get("aa", float("nan"))
aa_hdc  = m_hdc.get("aa", float("nan"))
af_ewc  = m_ewc.get("af", float("nan"))
af_hdc  = m_hdc.get("af", float("nan"))
ram_ewc_inf_ko  = m_ewc.get("ram_peak_bytes", 0) / 1024
ram_ewc_upd_ko  = mem_ewc_inline["update"]["ram_peak_bytes_update"] / 1024
ram_hdc_fp32_ko = m_hdc.get("estimated_ram_fp32_bytes", 0) / 1024
ram_hdc_int8_ko = m_hdc.get("estimated_ram_int8_bytes",  0) / 1024
lat_ewc = m_ewc.get("inference_latency_ms", float("nan"))
lat_hdc = m_hdc.get("inference_latency_ms", float("nan"))

hdc_ok   = "acceptable ✅" if aa_hdc > 0.85 else "insuffisant (< 85 %) — voir TODO(arnaud) ⚠️"
af_hdc_s = "nul par construction ✅" if af_hdc < 0.001 else f"{af_hdc:.4f} — vérifier implémentation ⚠️"

conclusion = f"""
## Conclusions — Sprint 2 (Dataset 2, 3 domaines)

### Résultats clés

| Modèle                      | AA     | AF     | RAM peak                                           | Latence  |
|-----------------------------|--------|--------|----------------------------------------------------|----------|
| EWC Online (M2)             | {aa_ewc:.4f} | {af_ewc:.4f} | {ram_ewc_inf_ko:.1f} Ko (inf) / {ram_ewc_upd_ko:.1f} Ko (update) | {lat_ewc:.3f} ms |
| HDC (M3)                    | {aa_hdc:.4f} | {af_hdc:.4f} | {ram_hdc_fp32_ko:.1f} Ko FP32 / {ram_hdc_int8_ko:.1f} Ko INT8 estimé | {lat_hdc:.3f} ms |
| Fine-tuning naïf            | {m_naive.get('aa', float('nan')):.4f} | {m_naive.get('af', float('nan')):.4f} | — | — |
| Joint training (upper bound)| {m_joint.get('aa', float('nan')):.4f} | N/A   | — | — |

### Triple gap

- **Gap 1** (données industrielles réelles) : ✅ Les deux modèles sont validés sur Dataset 2
  (Industrial Equipment Monitoring, 7 672 échantillons, 3 domaines réels : Pump → Turbine → Compressor).
  > ⚠️ FIXME(gap1) : validation sur FEMTO PRONOSTIA prévue Phase 2 (Gap 1 complet).

- **Gap 2** (< 100 Ko RAM, chiffres mesurés) : ✅ EWC : {ram_ewc_upd_ko:.1f} Ko peak (update, pire cas) —
  HDC : {ram_hdc_fp32_ko:.1f} Ko FP32 / {ram_hdc_int8_ko:.1f} Ko INT8 estimé — tous deux sous 64 Ko.

- **Gap 3** (quantification INT8 pendant l'entraînement incrémental) : ⬜ Non adressé ici.
  Prévu Sprint 4 — module `src/utils/quantization.py` + extension buffer UINT8 sur TinyOL.

### Observations

1. **Précision** : EWC ({aa_ewc:.2%}) > HDC ({aa_hdc:.2%}) — écart de {(aa_ewc - aa_hdc):.2%}.
   HDC est {hdc_ok}.
2. **Oubli catastrophique HDC** : AF = {af_hdc_s}.
   L'accumulation additive des prototypes garantit AF ≈ 0 par construction.
3. **Mémoire** : HDC ({ram_hdc_fp32_ko:.1f} Ko FP32) est plus frugal qu'EWC au pire cas
   ({ram_ewc_upd_ko:.1f} Ko lors de la mise à jour Fisher).
4. **Domain shift** : Les 3 domaines du Dataset 2 ont des distributions proches —
   l'oubli catastrophique est quasi-absent même pour le fine-tuning naïf (AF = {m_naive.get('af', 0):.4f}).
   Limitation documentée dans S109_exp001.md.
   > TODO(arnaud) : ajouter une courbe d'accuracy au fil du temps (sample-level) pour mieux
   > illustrer l'absence de forgetting sur ce dataset.
"""

display(Markdown(conclusion))


## Conclusions — Sprint 2 (Dataset 2, 3 domaines)

### Résultats clés

| Modèle                      | AA     | AF     | RAM peak                                           | Latence  |
|-----------------------------|--------|--------|----------------------------------------------------|----------|
| EWC Online (M2)             | 0.9824 | 0.0010 | 1.1 Ko (inf) / 6.7 Ko (update) | 0.036 ms |
| HDC (M3)                    | 0.8698 | 0.0000 | 14.0 Ko FP32 / 6.0 Ko INT8 estimé | 0.048 ms |
| Fine-tuning naïf            | 0.9811 | 0.0000 | — | — |
| Joint training (upper bound)| 0.9811 | N/A   | — | — |

### Triple gap

- **Gap 1** (données industrielles réelles) : ✅ Les deux modèles sont validés sur Dataset 2
  (Industrial Equipment Monitoring, 7 672 échantillons, 3 domaines réels : Pump → Turbine → Compressor).
  > ⚠️ FIXME(gap1) : validation sur FEMTO PRONOSTIA prévue Phase 2 (Gap 1 complet).

- **Gap 2** (< 100 Ko RAM, chiffres mesurés) : ✅ EWC : 6.7 Ko peak (update, pire cas) —
  HDC : 14.0 Ko FP32 / 6.0 Ko INT8 estimé — tous deux sous 64 Ko.

- **Gap 3** (quantification INT8 pendant l'entraînement incrémental) : ⬜ Non adressé ici.
  Prévu Sprint 4 — module `src/utils/quantization.py` + extension buffer UINT8 sur TinyOL.

### Observations

1. **Précision** : EWC (98.24%) > HDC (86.98%) — écart de 11.26%.
   HDC est acceptable ✅.
2. **Oubli catastrophique HDC** : AF = nul par construction ✅.
   L'accumulation additive des prototypes garantit AF ≈ 0 par construction.
3. **Mémoire** : HDC (14.0 Ko FP32) est plus frugal qu'EWC au pire cas
   (6.7 Ko lors de la mise à jour Fisher).
4. **Domain shift** : Les 3 domaines du Dataset 2 ont des distributions proches —
   l'oubli catastrophique est quasi-absent même pour le fine-tuning naïf (AF = 0.0000).
   Limitation documentée dans S109_exp001.md.
   > TODO(arnaud) : ajouter une courbe d'accuracy au fil du temps (sample-level) pour mieux
   > illustrer l'absence de forgetting sur ce dataset.
